# Inworld Voice Isolator 사용법: pip 한 줄로 녹음 잡음 제거, 실시간 모드, ElevenLabs Voice Isolator와 차이

위에서부터 차례로 실행해 보세요.

### 설치

In [ ]:
%%bash
python3 -m venv /content/v && . /content/v/bin/activate && time pip install inworld-voice-isolation
du -sh /content/v

### 오프라인 분리 (메모리가 작으면 청크를 줄임)

In [ ]:
%%bash
. /content/v/bin/activate && cd /content && time voice-isolate dl/sample_14.wav out_clean.wav --chunk-seconds 2 --device cpu
ls -l out_clean*.wav
ls -lL ~/.cache/huggingface/hub/models--inworld--voice-isolator/blobs/

### 샘플 받기

In [ ]:
%%bash
mkdir -p /content/dl && curl -sLo /content/dl/sample_14.wav https://raw.githubusercontent.com/inworld-ai/voice-isolator/main/demo/examples/sample_14.wav

### 파형·스펙트로그램 비교

In [ ]:
%%bash
cd /content && . /content/v/bin/activate && pip install -q matplotlib && cat > plot.py <<'E'
import soundfile as sf, numpy as np, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fs=[('input','dl/sample_14.wav'),('voice (offline)','out_clean.wav'),('background','out_clean_background.wav'),('voice (streaming)','out_stream.wav')]
fig,ax=plt.subplots(len(fs),2,figsize=(14,10))
for i,(n,p) in enumerate(fs):
    x,sr=sf.read(p)
    if x.ndim>1:x=x[:,0]
    ax[i,0].plot(np.arange(len(x))/sr,x,lw=.3);ax[i,0].set_ylim(-1,1);ax[i,0].set_title(n+' waveform')
    ax[i,1].specgram(x,NFFT=1024,Fs=sr,noverlap=512,cmap='magma',vmin=-130,vmax=-30);ax[i,1].set_title(n+' spectrogram')
plt.tight_layout();plt.savefig('compare.png',dpi=90)
E
python plot.py

### 스트리밍 push/flush

In [ ]:
%%bash
cd /content && . /content/v/bin/activate && cat > st.py <<'E'
import time, torch
from inworld import voice_isolation
live = voice_isolation.VoiceIsolator('inworld/voice-isolator', mode='streaming', lookahead_frames=2, device='cpu')
s = live.stream()
wav, sr = voice_isolation.load_audio('dl/sample_14.wav')
out=[]; lat=[]
for a in range(0, len(wav), 960):
    t0=time.time(); out.append(s.push(wav[a:a+960])); lat.append(time.time()-t0)
out.append(s.flush())
print('max push', max(lat), 'mean push', sum(lat)/len(lat), 'chunks', len(lat))
voice_isolation.save_audio('out_stream.wav', torch.cat(out), sr)
E
python st.py